# Week 4 — Give a multimodal model visual evidence

**Research task:** Ask a vision-capable model to describe change across two supplied interaction frames while separating visible evidence from interpretation.

**Python introduced:** file paths, ordered image lists, nested message content and structured visual descriptions.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session04/session04_multimodal_evidence.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.


In [ ]:
SESSION = "session04"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib.util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The course repository root could not be found. Start Jupyter from the "
            "GenAI_Soc2026 folder with: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib.util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Load the supplied image transport utility and identify two frames

Each `Path` stores a location, not the image pixels themselves. `display(Image(...))` lets the researcher inspect the frames before asking a model about them. The supplied transport utility later converts a file to the data-URL representation OpenRouter accepts; students need to explain its file input and transport output, not binary encoding.


In [ ]:
from src.genai_soc.media import image_to_data_url

ROUTE = "openrouter" if IN_COLAB else "ollama"  # Ollama is the local default
frame_1 = ROOT / "slides" / "session04" / "images" / "zidane_fine2_168.png"
frame_2 = ROOT / "slides" / "session04" / "images" / "zidane_fine2_169.png"
timestamps = ["00:00", "00:01"]
print(frame_1)
print(frame_2)
print("Both files exist:", frame_1.exists() and frame_2.exists())


## State the evidentiary boundary and the required JSON fields

`instruction` tells the model to report visible change and withhold identity, motive and cause. `schema` is a nested dictionary describing the required fields and their types. It can constrain the form of a return; it cannot guarantee that a supposedly visible claim is actually present in a frame.


In [ ]:
prompt = (
    "Compare these ordered frames. Describe only visible change. Do not name people, "
    "infer motive, or use remembered event knowledge. Return JSON with exactly "
    "observable_change, visible_evidence, and interpretation_withheld."
)
schema = {
    "type": "object",
    "properties": {
        "observable_change": {"type": "string"},
        "visible_evidence": {"type": "array", "items": {"type": "string"}},
        "interpretation_withheld": {"type": "string"},
    },
    "required": ["observable_change", "visible_evidence", "interpretation_withheld"],
    "additionalProperties": False,
}

## Send the images through the selected route

The message contains ordered text and image parts. The OpenRouter branch converts each path to a data URL; the Ollama branch supplies local path strings in its `images` field. Each branch requests the same schema and stores the returned JSON text in `raw_json`. Route-specific transport differs even though the research question is held fixed.


In [ ]:
if ROUTE == "openrouter":
    content = [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_1)}},
        {"type": "image_url", "image_url": {"url": image_to_data_url(frame_2)}},
    ]
    messages = [{"role": "user", "content": content}]
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL, messages=messages, temperature=0,
            response_format={"type": "json_schema", "json_schema": {
                "name": "visible_sequence", "strict": True, "schema": schema,
            }},
        )
    raw_output = response.choices[0].message.content
else:
    messages = [{
        "role": "user", "content": prompt,
        "images": [str(frame_1), str(frame_2)],
    }]
    response = ollama.chat(
        model=LOCAL_MODEL, messages=messages, format=schema,
        options={"temperature": 0},
    )
    raw_output = response.message.content

print("Raw JSON text:", raw_output)

## Parse the three fields and compare them with the images

`json.loads(...)` creates a dictionary, then three key lookups retrieve the model's description, claimed visible evidence and withheld interpretation. Printing those values makes them available for frame-by-frame checking. A well-formed list of visible evidence is not proof that each item is visible.


In [ ]:
description = json.loads(raw_output)
print("Observable change:", description["observable_change"])
print("Visible evidence:", description["visible_evidence"])
print("Interpretation withheld:", description["interpretation_withheld"])

# ONE CHANGE: replace frame_2 with zidane_fine2_170.png and rerun.

## Methodological check

The model can still recognize a famous event or import a motive despite the constraint. Each returned claim must be checked against pixels in the two displayed frames.
## Completion recording

Use one route, replace the second frame with `zidane_fine2_170.png` and rerun. Explain each path, the image representation used by your route, the raw JSON and all three fields. Identify anything not visibly supported.

Explain every input and output aloud. Never show the shared key.